In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import polars as pl
import plotly.express as px
import seaborn as sns
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from utilsforecast.losses import *
from utilsforecast.evaluation import evaluate
from plotly.subplots import make_subplots
import pandas as pd
from sktime.transformations.series.impute import Imputer
from plotting_utils import (
    plotly_series as plot_series,
    plot_data_availability_heatmap,
    plot_missing_percentage,
)
import numpy as np

import plotly.graph_objects as go


In [ ]:
from functools import partial

metrics = [mae]

# Introduction: Handling Missing and Abnormal Data in Time Series Forecasting

Time series data, which records values sequentially over time, is the backbone of many forecasting applications—ranging from weather prediction to energy consumption analysis. However, real-world time series datasets are rarely perfect. They often contain **missing values** (gaps where data was not recorded) or **abnormal values** (outliers or errors due to faulty sensors, data entry mistakes, or unexpected events).

Understanding and addressing these issues is crucial for building accurate and reliable forecasting models. If left untreated, missing or abnormal data can lead to:

- **Biased model training:** Models may learn incorrect patterns or relationships.
- **Reduced forecast accuracy:** Predictions may be less reliable or even misleading.
- **Misleading evaluation metrics:** Performance measures may not reflect true model capability.

## Why Do Missing or Abnormal Values Occur?

- **Sensor failures or communication errors** can lead to missing data points.
- **Manual data entry mistakes** or system glitches may introduce outliers.
- **Natural anomalies** (like sudden weather changes or holidays) can cause legitimate but rare spikes or drops.

## The Importance of Data Cleaning

Before applying any forecasting technique, it is essential to:

1. **Detect** missing and abnormal values.
2. **Decide** on an appropriate strategy to handle them (e.g., imputation, removal, or correction).
3. **Apply** these strategies consistently to ensure the integrity of the time series.

By carefully handling missing and abnormal data, we lay a strong foundation for robust time series analysis and forecasting. In the following sections, we will explore practical methods for detecting and addressing these issues, using real-world examples and step-by-step explanations.

In [ ]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

In [ ]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [ ]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select(
        [
            time_,
            id_,
            target_,
            "Acorn",
            "Acorn_grouped",
            "holidays",
            "visibility",
            "windBearing",
            "temperature",
            "dewPoint",
            "pressure",
            "apparentTemperature",
            "windSpeed",
            "precipType",
            "icon",
            "humidity",
            "summary",
        ]
    )
    .explode(
        [
            time_,
            target_,
            "holidays",
            "visibility",
            "windBearing",
            "temperature",
            "dewPoint",
            "pressure",
            "apparentTemperature",
            "windSpeed",
            "precipType",
            "icon",
            "humidity",
            "summary",
        ]
    )
)
data.head()

## Detecting, Deciding, and Applying Strategies for Missing Data in Time Series

### 1. Detecting Missing Data

The first step in handling missing data is **detection**. In time series, missing values can occur as:

- **Explicit nulls:** Data points are present but marked as `null`, `NaN`, or similar.
- **Implicit gaps:** Expected timestamps are missing entirely from the dataset.

**Why is this important?**  
Missing data can distort trends, seasonality, and other patterns crucial for accurate forecasting.

**How do we detect missing data?**

- **Check for nulls:** Scan for `null` or `NaN` values in the target variable.
- **Check for time gaps:** Compare the actual timestamps to the expected frequency (e.g., every 30 minutes). Missing timestamps indicate gaps.

*Example:*  
If our data should have a reading every 30 minutes, but some timestamps are missing, we have implicit missing data.

### 2. Deciding on a Strategy

Once missing data is detected, we must **decide** how to handle it. Common strategies include:

- **Imputation:** Fill missing values using statistical methods (e.g., forward fill, backward fill, mean, interpolation).
- **Removal:** Drop rows with missing values (only if missingness is rare and random).
- **Leave as-is:** Some models can handle missing values natively, but this is less common.

**How to choose?**

- If missingness is random and infrequent, imputation is often preferred.
- If large blocks are missing, imputation may introduce bias—removal or more advanced methods may be needed.
- Always consider the impact on downstream analysis and model performance.

### 3. Applying the Strategy

After deciding, we **apply** the chosen method:

- **Forward fill:** Replace missing values with the last known value.
- **Backward fill:** Replace missing values with the next known value.
- **Interpolation:** Estimate missing values based on surrounding data.
- **Custom imputation:** Use domain knowledge or advanced algorithms.

---

**Key Takeaway:**  
Detecting and thoughtfully handling missing data is essential for trustworthy time series forecasting. By systematically detecting, deciding, and applying the right strategy, we ensure our models are trained on clean, reliable data—setting the stage for accurate predictions.

In [ ]:
id_orders = data.select(pl.col(id_)).unique()

In [ ]:
plot_missing_percentage(data, id_orders=id_orders)

In [ ]:
plot_data_availability_heatmap(data, id_orders=id_orders)

## Handling Time Series with High Proportions of Missing Data (>20%)

When analyzing time series data, encountering series with a large proportion of missing values—such as more than 20%—is a significant challenge. These gaps can undermine the reliability of your analysis and the accuracy of your forecasts. Let’s explore why this is problematic and discuss practical strategies for handling such cases.

---

### Why Is High Missingness a Problem?

- **Loss of Information:** Large gaps mean less data to learn from, making it harder to detect trends, seasonality, or anomalies.
- **Imputation Bias:** Standard imputation methods (like forward fill or interpolation) may introduce bias or unrealistic values when applied to long stretches of missing data.
- **Model Instability:** Forecasting models may become unstable or overfit to the limited available data, reducing generalizability.

---

### Approaches for Handling High-Missingness Series

#### 1. **Assess the Importance of Each Series**

- **Business Relevance:** Is the series critical for your analysis or decision-making? If not, consider excluding it.
- **Data Coverage:** Sometimes, it’s better to focus on series with more complete data for robust modeling.

#### 2. **Remove or Exclude Series**

- **Thresholding:** Set a threshold (e.g., remove any series with more than 20% missing data).
- **Pros:** Ensures your analysis is based on reliable data.
- **Cons:** You may lose potentially valuable information.

#### 3. **Advanced Imputation Techniques**

For series you must keep, consider more sophisticated imputation methods:

- **Model-Based Imputation:** Use models (e.g., ARIMA, k-NN, or machine learning) trained on similar series to predict missing values.
- **Multiple Imputation:** Generate several plausible imputed datasets and combine results to account for uncertainty.
- **Seasonal Imputation:** If strong seasonality exists, fill gaps using values from the same season in other years.

**Caution:**  
Imputation over long gaps can introduce significant uncertainty. Always visualize imputed values and, if possible, validate with domain knowledge.

#### 4. **Leverage Cross-Series Information**

- **Panel Data Models:** If you have many similar time series (e.g., from different households), use models that borrow strength across series, such as hierarchical or mixed-effects models.
- **Clustering:** Group similar series and use the group’s pattern to inform imputation.

#### 5. **Flag and Treat with Caution**

- **Flag Imputed Values:** Mark which values are imputed so downstream analyses can account for increased uncertainty.
- **Sensitivity Analysis:** Compare results with and without high-missingness series to assess their impact.

---

### Practical Example

Suppose you have a time series with 30% missing data, including long consecutive gaps. Instead of simply forward-filling, you might:

- Use a model trained on other, more complete series to predict missing values.
- Impute missing values based on the average pattern of similar series (e.g., households in the same neighborhood).
- Exclude the series if it’s not essential, documenting your rationale.

---

### Key Takeaways

- **High proportions of missing data require careful handling.**
- **Simple imputation may not be sufficient—consider advanced or cross-series methods.**
- **Always document your approach and assess the impact of missing data on your results.**

By thoughtfully addressing high-missingness series, you ensure your time series analysis remains robust, transparent, and trustworthy.

## Excluding Time Series with Excessive Missing Data

When a time series contains a high proportion of missing values (for example, more than 20%), it is often best practice to exclude it from further analysis and modeling. This approach helps ensure the reliability and accuracy of your forecasts.

### Why Exclude High-Missingness Series?

- **Insufficient Information:** Large gaps reduce the amount of usable data, making it difficult for models to learn meaningful patterns.
- **Imputation Risks:** Filling long stretches of missing data can introduce bias or unrealistic values, especially if the missingness is not random.
- **Model Stability:** Including incomplete series can destabilize model training and degrade overall performance.

### Practical Steps

1. **Set a Threshold:**  
    Decide on a cutoff (e.g., exclude any series with more than 20% missing data).

2. **Filter the Data:**  
    Remove these series from your dataset before proceeding with imputation or modeling.

3. **Document Your Decision:**  
    Clearly state which series were excluded and why, to maintain transparency in your analysis.

### Example

Suppose your dataset contains 50 electricity meter time series, and 5 of them have more than 20% missing values. You would:

- Identify these series using the missing data percentage.
- Exclude them from your working dataset.
- Proceed with cleaning and modeling only on the remaining, more complete series.

**Key Principle:**  
By focusing on time series with sufficient data, you improve the robustness and trustworthiness of your forecasting results. Always document your exclusion criteria and consider the potential impact on your analysis.

In [ ]:
data = data.filter(
    pl.col(target_).is_not_null().sum().truediv(pl.len()).over(id_).ge(0.8)
)

In [ ]:
plot_missing_percentage(data)

In [ ]:
plot_data_availability_heatmap(data)

## Assessing Data Quality After Exclusion and Introducing Imputation Experiments

Now that we've excluded time series with excessive missing data (greater than 20%), let's take a moment to assess the quality of our remaining dataset.

### Improved Data Quality

By filtering out highly incomplete series, our dataset now consists of time series with much more reliable and consistent data coverage. This means:

- **Fewer large gaps:** Most series now have only small, sporadic missing values.
- **Better trend and seasonality detection:** With more complete data, patterns are easier to identify and model.
- **Greater confidence in imputation:** Imputation methods are more likely to produce realistic values when only short gaps need to be filled.

If you revisit the missing data visualizations (such as the percentage bar plot and the data availability heatmap), you'll notice that the remaining series have much lower proportions of missing values, and the gaps are generally shorter and less frequent.

---

## Exploring Imputation Techniques: A Hands-On Approach

To truly understand the strengths and weaknesses of different imputation methods, it's helpful to **experiment**:

1. **Artificially remove (mask) some known values** from a time series.
2. **Apply various imputation techniques** to fill these artificial gaps.
3. **Compare the imputed values to the original (true) values** using a metric such as Mean Absolute Error (MAE).

This approach allows us to objectively evaluate how well each imputation method recovers the true data.

### Why Do This?

- **Realistic benchmarking:** Since we know the true values we've masked, we can directly measure imputation accuracy.
- **Method selection:** Some techniques may work better for your data's characteristics (e.g., seasonality, volatility).
- **Building intuition:** Seeing the results helps you understand the practical trade-offs of each method.

---

### Next Steps

In the following sections, we'll:

- Select a representative time series from our cleaned dataset.
- Randomly mask a small percentage of its values to simulate missing data.
- Apply several imputation strategies (forward fill, backward fill, linear interpolation, and seasonal imputation).
- Calculate the Mean Absolute Error (MAE) for each method to compare their performance.

This hands-on experiment will give you a clear, data-driven understanding of how different imputation techniques perform on real electricity load demand data.

---

**Mathematical Note:**  
The Mean Absolute Error (MAE) is defined as:

$$
\mathrm{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|
$$

where $y_i$ is the true value and $\hat{y}_i$ is the imputed value at position $i$.

---

Let's get started with the experiment!

In [ ]:
selected_ids = [
    "MAC000647",
    "MAC000193",
    "MAC000194",
    "MAC000289",
    "MAC000317",
    "MAC000632",
    "MAC000647",
    "MAC001825",
    "MAC001826",
    "MAC004628",
    "MAC004637",
]
print(f"Selected {len(selected_ids)} IDs for analysis: {selected_ids}")

In [ ]:
selected_one_date = pd.to_datetime("2013-07-04")
selected_plus_2w = selected_one_date + pd.Timedelta(weeks=2)
selected_minus_2w = selected_one_date - pd.Timedelta(weeks=2)
selected_data = data.filter(
    (pl.col(id_).is_in(selected_ids))
    & (pl.col(time_).ge(selected_minus_2w))
    & (pl.col(time_).le(selected_plus_2w))
).select(time_, id_, target_)

In [ ]:
plot_data_availability_heatmap(selected_data)

In [ ]:
plot_series(selected_data)

In [ ]:
# Mask the target variable 'y' on the selected date by setting it to null in a new column 'y_masked'
selected_data = selected_data.with_columns(
    pl.when(time_col.dt.date() == selected_one_date.date())
    .then(None)  # Set to null if the date matches
    .otherwise(target_col)  # Otherwise, keep the original value
    .alias("y_masked")
)

## Imputation Strategies for Handling Missing Data in Electricity Load Demand

When working with electricity load demand time series, missing data is a common challenge. Accurate imputation—filling in these gaps—is crucial for maintaining the integrity of your analysis and forecasts. Let's explore the most widely used imputation strategies, their rationale, and practical considerations.

### 1. Forward Fill (Last Observation Carried Forward)

**How it works:**  
Each missing value is replaced with the most recent observed value.

- **When to use:**  
    - Short gaps in data.
    - When electricity usage is relatively stable over short periods.

- **Pros:**  
    - Simple and fast.
    - Preserves recent trends.

- **Cons:**  
    - Can propagate errors if a long sequence is missing.
    - May not capture sudden changes in demand.

**Mathematical notation:**  
If $y_t$ is missing, set $y_t = y_{t-1}$.

---


In [ ]:
selected_data = selected_data.sort(
    [id_, time_]
).with_columns(  # Ensure data is sorted by household and time
    pl.col("y_masked")
    .forward_fill()
    .over(id_)  # Apply forward fill within each household
    .alias("forward fill")
)

# Let's preview the result to see how missing values were filled
imputed = (
    selected_data.select([c for c in selected_data.columns if c != "y_masked"])
    .sort([id_, time_])
    .filter(time_col.dt.date() == selected_one_date.date())
)

In [ ]:
plot_series(imputed, imputed)

In [ ]:
evaluate(imputed, metrics=metrics)


### 2. Backward Fill (Next Observation Carried Backward)

**How it works:**  
Each missing value is replaced with the next available value.

- **When to use:**  
    - At the start of a series or when forward fill is not appropriate.

- **Pros:**  
    - Simple to implement.

- **Cons:**  
    - May introduce future information into the past (data leakage).

**Mathematical notation:**  
If $y_t$ is missing, set $y_t = y_{t+1}$.

---


In [ ]:
selected_data = selected_data.sort(
    [id_, time_]
).with_columns(  # Ensure data is sorted by household and time
    pl.col("y_masked")
    .backward_fill()
    .over(id_)  # Apply backward fill within each household
    .alias("backward fill")
)

# Let's preview the result to see how missing values were filled
imputed = (
    selected_data.select([c for c in selected_data.columns if c != "y_masked"])
    .sort([id_, time_])
    .filter(time_col.dt.date() == selected_one_date.date())
)
fig = plot_series(imputed, imputed, models=["backward fill"])
fig.show()
evaluate(imputed, metrics=metrics)


### 3. Linear Interpolation

**How it works:**  
Estimates missing values by drawing a straight line between the previous and next observed values.

- **When to use:**  
    - When demand changes smoothly over time.
    - For short to moderate gaps.

- **Pros:**  
    - Captures gradual changes.
    - More realistic than simple forward/backward fill.

- **Cons:**  
    - Not suitable for abrupt changes or highly volatile periods.

**Mathematical notation:**  
If $y_t$ is missing between $y_{t-1}$ and $y_{t+k}$:
$$
y_t = y_{t-1} + \frac{y_{t+k} - y_{t-1}}{k} \cdot (t - (t-1))
$$

---


In [ ]:
selected_data = selected_data.sort(
    [id_, time_]
).with_columns(  # Ensure data is sorted by household and time
    pl.col("y_masked")
    .interpolate()
    .over(id_)  # Apply interpolation within each household
    .alias("interpolated")
)

# Let's preview the result to see how missing values were filled
imputed = (
    selected_data.select([c for c in selected_data.columns if c != "y_masked"])
    .sort([id_, time_])
    .filter(time_col.dt.date() == selected_one_date.date())
)
fig = plot_series(imputed, imputed, models=["interpolated"])
fig.show()
evaluate(imputed, metrics=metrics)


### 4. Seasonal Imputation

**How it works:**  
Fills missing values using data from the same time in previous cycles (e.g., same hour last week).

- **When to use:**  
    - Strong daily or weekly seasonality in electricity demand.

- **Pros:**  
    - Respects recurring patterns.
    - Useful for regular, predictable demand cycles.

- **Cons:**  
    - Requires enough historical data.
    - Less effective if seasonality is weak or changing.

**Mathematical notation:**  
If $y_t$ is missing, set $y_t = y_{t-s}$, where $s$ is the seasonal period (e.g., $s=48$ for 30-minute intervals in a day).

---


In [ ]:
selected_data = selected_data.sort(
    [id_, time_]
).with_columns(  # Ensure data is sorted by household and time
    pl.col("y_masked")
    .fill_null(
        pl.col(target_).shift(48).over(id_)
    )  # Fill nulls with the value from 48 hours ago
    .alias("daily forward fill")
)

# Let's preview the result to see how missing values were filled
imputed = (
    selected_data.select([c for c in selected_data.columns if c != "y_masked"])
    .sort([id_, time_])
    .filter(time_col.dt.date() == selected_one_date.date())
)
fig = plot_series(imputed, imputed, models=["daily forward fill"])
fig.show()
evaluate(imputed, metrics=metrics)

In [ ]:
selected_data = selected_data.sort(
    [id_, time_]
).with_columns(  # Ensure data is sorted by household and time
    pl.col("y_masked")
    .fill_null(
        pl.col(target_).shift(-48).over(id_)
    )  # Fill nulls with the value from 48 hours ago
    .alias("daily backward fill")
)

# Let's preview the result to see how missing values were filled
imputed = (
    selected_data.select([c for c in selected_data.columns if c != "y_masked"])
    .sort([id_, time_])
    .filter(time_col.dt.date() == selected_one_date.date())
)
fig = plot_series(imputed, imputed, models=["daily backward fill"])
fig.show()
evaluate(imputed, metrics=metrics)

### Caution: Risks of Data Leakage and Bias with Imputation Methods

When applying imputation strategies like **backward fill** and **forward fill** in time series analysis, it's crucial to understand their potential pitfalls—especially regarding **data leakage** and **evaluation bias**.

---

#### Backward Fill and Data Leakage

**Backward fill** replaces missing values with the next available observation in the series. While this can be useful, it poses a significant risk when splitting your data into **training** and **testing** sets for forecasting:

- **What is data leakage?**  
    Data leakage occurs when information from the future (test set) unintentionally influences the training process, leading to overly optimistic performance estimates.

- **How does backward fill cause leakage?**  
    If you apply backward fill before splitting your data, missing values in the training set may be filled using values from the test set—effectively giving your model a "peek" into the future.

- **Best practice:**  
    **Always split your data into train and test sets before applying backward fill.** This ensures that imputation only uses information available up to that point in time, preserving the integrity of your evaluation.

---

#### Forward Fill and Evaluation Bias

**Forward fill** replaces missing values with the most recent past observation. While this method avoids future data leakage, it can still introduce **bias** in your evaluation metrics:

- **Why does this happen?**  
    If large blocks of missing values are filled with the same previous value, your imputed series may appear artificially "smooth" or "stable." This can make your model's predictions seem more accurate than they truly are, especially if your evaluation metric (like MAE) is calculated on these imputed values.

- **Example:**  
    Suppose a long gap is filled with a constant value. If your model predicts this constant, the error will be zero, even though the true underlying values may have varied significantly.

- **Best practice:**  
    **Be cautious when interpreting evaluation metrics on imputed data.** Consider masking imputed values during evaluation, or using more sophisticated imputation methods for long gaps.

---

#### Key Takeaways

- **Backward fill can cause data leakage if applied before splitting into train/test.**
- **Forward fill can bias evaluation metrics, especially over long gaps.**
- **Always apply imputation methods thoughtfully, respecting the temporal order of your data.**
- **Document your imputation strategy and its potential impact on your results.**

By understanding these risks, you can make more informed decisions and ensure the validity of your time series analysis and forecasting.

In [ ]:
selected_data = selected_data.sort(
    [id_, time_]
).with_columns(  # Ensure data is sorted by household and time
    pl.col("y_masked")
    .fill_null(
        pl.col(target_).shift(48 * 7).over(id_)
    )  # Fill nulls with the value from 48*7 hours ago
    .alias("weekly forward fill")
)

# Let's preview the result to see how missing values were filled
imputed = (
    selected_data.select([c for c in selected_data.columns if c != "y_masked"])
    .sort([id_, time_])
    .filter(time_col.dt.date() == selected_one_date.date())
)
fig = plot_series(imputed, imputed, models=["weekly forward fill"])
fig.show()
evaluate(imputed, metrics=metrics)

In [ ]:
selected_data = selected_data.sort(
    [id_, time_]
).with_columns(  # Ensure data is sorted by household and time
    pl.col("y_masked")
    .fill_null(pl.col(target_).mean().over([id_], pl.col(time_).dt.hour()))
    .alias("hourly mean fill")
)

# Let's preview the result to see how missing values were filled
imputed = (
    selected_data.select([c for c in selected_data.columns if c != "y_masked"])
    .sort([id_, time_])
    .filter(time_col.dt.date() == selected_one_date.date())
)
fig = plot_series(imputed, imputed, models=["hourly mean fill"])
fig.show()
evaluate(imputed, metrics=metrics)


### 5. Model-Based Imputation

**How it works:**  
Uses statistical or machine learning models (e.g., ARIMA, k-NN, regression) to predict missing values based on observed data.

- **When to use:**  
    - Complex missing patterns.
    - When other methods are insufficient.

- **Pros:**  
    - Can capture complex relationships and trends.

- **Cons:**  
    - Requires more computation and expertise.
    - Risk of overfitting if not carefully validated.

---


In [ ]:
imputer = Imputer(method="drift")
imputed = imputer.fit_transform(
    selected_data.to_pandas().set_index([id_, time_])[["y_masked"]]
)

selected_data = selected_data.sort(
    [id_, time_]
).with_columns(  # Ensure data is sorted by household and time
    pl.Series(values=imputed["y_masked"], name="drift imputed")
)

# Let's preview the result to see how missing values were filled
imputed = (
    selected_data.select([c for c in selected_data.columns if c != "y_masked"])
    .sort([id_, time_])
    .filter(time_col.dt.date() == selected_one_date.date())
)
fig = plot_series(imputed, imputed, models=["drift imputed"])
fig.show()
evaluate(imputed, metrics=metrics)

In [ ]:
from sktime.forecasting.statsforecast import StatsForecastAutoETS

In [ ]:
imputer = Imputer(
    method="forecaster",
    forecaster=StatsForecastAutoETS(season_length=48, model="ANA", damped=False),
)
imputed = imputer.fit_transform(
    selected_data.to_pandas().set_index([id_, time_])[["y_masked"]]
)

selected_data = selected_data.sort(
    [id_, time_]
).with_columns(  # Ensure data is sorted by household and time
    pl.Series(values=imputed["y_masked"], name="ets imputed")
)

# Let's preview the result to see how missing values were filled
imputed = (
    selected_data.select([c for c in selected_data.columns if c != "y_masked"])
    .sort([id_, time_])
    .filter(time_col.dt.date() == selected_one_date.date())
)
fig = plot_series(imputed, imputed, models=["ets imputed"])
fig.show()
evaluate(imputed, metrics=metrics)


### Choosing the Right Strategy

- **Short, random gaps:** Forward/backward fill or linear interpolation.
- **Seasonal patterns:** Seasonal imputation.
- **Complex or structured missingness:** Model-based approaches.

**Key Principle:**  
Always visualize and understand your data before choosing an imputation method. Test different strategies and evaluate their impact on downstream forecasting performance.


## Conclusion: Best Imputation Strategies for Electricity Load Demand Forecasting

After systematically experimenting with various imputation techniques—including forward fill, backward fill, linear interpolation, daily/weekly seasonal imputation, and model-based approaches—we can draw some practical conclusions for electricity load demand time series:

### Key Findings

- **Seasonal Imputation (Daily and Weekly):**
    - These methods leverage the strong recurring patterns in electricity usage, such as daily and weekly cycles.
    - By filling missing values with data from the same time on previous days or weeks, we respect the natural seasonality of the data.
    - In our experiments, both daily and weekly seasonal imputation produced more realistic and accurate results compared to simpler methods.

- **Seasonal Model-Based Imputation:**
    - Advanced models that explicitly account for seasonality (like ETS or other time series models) also performed well.
    - These approaches can capture complex seasonal and trend components, making them robust for structured missingness.

- **Simplicity and Effectiveness:**
    - While model-based methods are powerful, they require more computation and expertise.
    - **Weekly seasonal imputation** offers an excellent balance between simplicity and effectiveness, making it a practical default for many electricity demand forecasting tasks.

### Practical Recommendation

**For electricity load demand forecasting, we recommend using weekly seasonal imputation as the default strategy for handling missing data.**  
This approach is easy to implement, leverages the natural weekly cycles in demand, and has demonstrated strong performance in our hands-on evaluation.

---

**Next Steps:**  
Going forward, we will apply weekly seasonal imputation to handle missing values in our time series. This will ensure our data is both clean and seasonally consistent, providing a solid foundation for accurate forecasting.

---

*Remember: Always visualize your imputed data and, when possible, validate your approach with domain knowledge or additional experiments!*